# Spectral Analysis — Phase 6: HotpotQA Generalization + Baseline Comparison

**Goal:** Two-part analysis:
- **Part 1 (static)** — load Phase 4/5 pkl results, compare our numbers against RENT, LapEigvals, LOS-Net.
- **Part 2 (new inference)** — run Mistral-7B-Instruct-v0.2 on HotpotQA to directly benchmark against LOS-Net (AUC=72.92%).

**Design decisions (Step 62):**
1. **No trace/answer split** — full response only. The 'Answer:' marker appeared 0–2% of the time; the fallback (last 25%) is arbitrary. Full 50–200 token response IS the signal.
2. **Window ablation** — sw_var_peak tested with w ∈ {3, 5, 7, 9, 16}, sw_step=1. LSC paper (arXiv:2601.19918) confirms w=2–3 optimal on short factual QA.
3. HotpotQA multi-hop reasoning produces step-level entropy structure (unlike single-hop TriviaQA) — better transfer candidate for spectral features.

**Primary comparison:** best fusion AUC vs LOS-Net 72.92% on HotpotQA/Mistral-7B.

**Decision gates G0–G6** with automatic pass/fail are in Cell 12.

In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', '-q', 'transformers>=4.40', 'datasets', 'accelerate', 'scipy'], check=True)

try:
    from huggingface_hub import login
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print('HuggingFace login OK.')
except Exception as e:
    print(f'HF login skipped: {e}')

print('Ready.')

Mounted at /content/drive
HuggingFace login OK.
Ready.


In [2]:
import os, gc, re, itertools, pickle
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from scipy.linalg import eigh
from scipy.signal import stft as scipy_stft
from tqdm import tqdm

# ── I/O ────────────────────────────────────────────────────────────────────────
def load_cache(path):
    return pickle.load(open(path, 'rb')) if os.path.exists(path) else {}

def save_cache(obj, path):
    pickle.dump(obj, open(path, 'wb'))

def free_memory():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

# ── Model loading / generation ─────────────────────────────────────────────────
def load_model(model_id):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id, device_map='auto', torch_dtype=torch.float16, trust_remote_code=True)
    mdl.eval()
    print(f'Loaded {model_id}')
    return mdl, tok

def fmt_prompt(tok, msg):
    try:
        return tok.apply_chat_template([{'role': 'user', 'content': msg}],
                                       tokenize=False, add_generation_prompt=True)
    except:
        return f'<|user|>\n{msg}\n<|assistant|>\n'

def token_entropies_from_scores(scores, K=15):
    ents = []
    for s in scores:
        lp   = F.log_softmax(s[0], dim=-1)
        topk = torch.topk(lp, min(K, lp.shape[-1])).values
        p    = torch.exp(topk); p = p / (p.sum() + 1e-12)
        ents.append(-(p * torch.log(p + 1e-12)).sum().item())
    return ents

def generate_full(mdl, tok, prompt_msg, temperature=1.0, K=15, max_new_tokens=512):
    prompt = fmt_prompt(tok, prompt_msg)
    inputs = tok(prompt, return_tensors='pt').to(mdl.device)
    if 'token_type_ids' in inputs: del inputs['token_type_ids']
    with torch.no_grad():
        out = mdl.generate(**inputs, max_new_tokens=max_new_tokens,
                           do_sample=True, temperature=temperature, top_k=50,
                           output_scores=True, return_dict_in_generate=True,
                           pad_token_id=tok.eos_token_id)
    gen_ids   = out.sequences[0][inputs.input_ids.shape[1]:]
    full_text = tok.decode(gen_ids, skip_special_tokens=True).strip()
    all_ents  = token_entropies_from_scores(out.scores, K)
    return full_text, all_ents

# ── Statistics ─────────────────────────────────────────────────────────────────
def boot_auc(y, scores, n=1000):
    y, s = np.array(y), np.array(scores)
    if len(np.unique(y)) < 2 or np.std(s) < 1e-8:
        return float('nan'), float('nan'), float('nan')
    base = roc_auc_score(y, s)
    rng  = np.random.default_rng(42)
    boots = []
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2: continue
        boots.append(roc_auc_score(y[idx], s[idx]))
    lo, hi = np.percentile(boots, [2.5, 97.5]) if boots else (base, base)
    return base, lo, hi

def nadler_fuse(*views):
    X = np.column_stack(views); n, k = X.shape
    C = np.cov(X.T)
    if C.ndim == 0: C = np.array([[float(C)]])
    try:
        off = C - np.diag(np.diag(C))
        rs, cs = off.sum(1), off.sum(0)
        M = np.diag(rs) @ np.linalg.pinv(C) @ np.diag(cs)
        _, vecs = eigh(M)
        w = np.abs(vecs[:, -1]); w /= w.sum() + 1e-12
    except Exception:
        w = np.ones(k) / k
    return X @ w, w

def best_nadler_on(feats_dict, feat_names, labels_, max_size=3, label=''):
    auc_m, sign_m = {}, {}
    for n_ in feat_names:
        ap, *_ = boot_auc(labels_,  feats_dict[n_])
        an, *_ = boot_auc(labels_, -feats_dict[n_])
        if ap >= an: auc_m[n_], sign_m[n_] = ap, +1
        else:        auc_m[n_], sign_m[n_] = an, -1
    oriented = {n_: feats_dict[n_] * sign_m[n_] for n_ in feat_names}
    rho = {}
    for a, b in itertools.combinations_with_replacement(feat_names, 2):
        r, _ = spearmanr(oriented[a], oriented[b])
        rho[(a, b)] = rho[(b, a)] = r
    info = [n_ for n_ in feat_names if auc_m[n_] > 0.50]
    total_combos = sum(
        sum(1 for _ in itertools.combinations(info, size))
        for size in range(2, min(len(info) + 1, max_size + 1))
    )
    print(f'  [{label}] {len(feat_names)} features, {len(info)} informative, '
          f'max_size={max_size} → {total_combos} raw combos')
    best_a, best_lo, best_hi, best_s = 0.0, 0.0, 0.0, None
    checked, skipped = 0, 0
    for size in range(2, min(len(info) + 1, max_size + 1)):
        size_combos  = list(itertools.combinations(info, size))
        valid_in_size = 0
        for s in size_combos:
            if any(abs(rho[(a, b)]) >= 0.75 for a, b in itertools.combinations(s, 2)):
                skipped += 1; continue
            fused, _ = nadler_fuse(*[oriented[n_] for n_ in s])
            a, lo, hi = boot_auc(labels_, fused)
            if a > best_a: best_a, best_lo, best_hi, best_s = a, lo, hi, s
            checked += 1; valid_in_size += 1
        print(f'    size={size}: {len(size_combos)} combos, {valid_in_size} passed ρ-filter, '
              f'best so far={100*best_a:.1f}%')
    print(f'  [{label}] done — checked={checked}, skipped(ρ)={skipped}, best={100*best_a:.1f}%')
    return best_a, best_lo, best_hi, best_s

# ── Feature extraction (identical to Phase 5) ──────────────────────────────────
def compute_spectral_features(ents, min_len=8):
    e = np.array(ents, dtype=float); N = len(e)
    if N < min_len: return None
    e_ac     = e - e.mean()
    fft_vals = np.fft.rfft(e_ac)
    psd      = np.abs(fft_vals) ** 2
    freqs    = np.fft.rfftfreq(N)
    psd_sum  = psd.sum() + 1e-12
    psd_norm = psd / psd_sum
    spec_ent  = -np.sum(psd_norm * np.log(psd_norm + 1e-12))
    low_mask  = (freqs > 0.0) & (freqs <= 0.10)
    high_mask = (freqs >= 0.40) & (freqs <= 0.50)
    low_power  = psd[low_mask].sum()  / psd_sum
    high_power = psd[high_mask].sum() / psd_sum
    hl_ratio   = high_power / (low_power + 1e-12)
    ac_mask    = freqs > 0
    dom_freq   = float(freqs[ac_mask][np.argmax(psd[ac_mask])]) if ac_mask.sum() > 0 else 0.0
    centroid   = float(np.sum(freqs[ac_mask] * psd_norm[ac_mask]) /
                       (psd_norm[ac_mask].sum() + 1e-12)) if ac_mask.sum() > 0 else 0.0
    return {'spectral_entropy': float(spec_ent), 'low_band_power': float(low_power),
            'high_band_power': float(high_power), 'hl_ratio': float(hl_ratio),
            'dominant_freq': dom_freq, 'spectral_centroid': centroid}

def compute_stft_features(ents, nperseg=16, noverlap=8, min_len=32):
    e = np.array(ents, dtype=float)
    if len(e) < min_len:
        return {'stft_max_high_power': 0.0, 'stft_spectral_entropy': 0.0}
    e_ac = e - e.mean()
    f, t, Zxx = scipy_stft(e_ac, nperseg=nperseg, noverlap=noverlap)
    psd = np.abs(Zxx) ** 2
    high_mask = f >= 0.40
    if high_mask.sum() > 0 and psd.shape[1] > 0:
        high_frac = psd[high_mask].sum(0) / (psd.sum(0) + 1e-12)
        max_local_high = float(high_frac.max())
    else:
        max_local_high = 0.0
    psd_n     = psd / (psd.sum(0, keepdims=True) + 1e-12)
    frame_ent = -np.sum(psd_n * np.log(psd_n + 1e-12), axis=0)
    stft_ent  = float(frame_ent.mean()) if len(frame_ent) > 0 else 0.0
    return {'stft_max_high_power': max_local_high, 'stft_spectral_entropy': stft_ent}

def compute_time_domain(ents, tail_frac=0.20, sw_window=16, sw_step=1):
    e  = np.array(ents, dtype=float)
    W  = max(1, int(len(e) * tail_frac))
    rpdi = float(e[-W:].mean() / (e.mean() + 1e-12))
    if len(e) >= sw_window:
        sw_vars = [np.var(e[i:i+sw_window])
                   for i in range(0, len(e)-sw_window+1, sw_step)]
        sw_var_peak = float(np.max(sw_vars))
    else:
        sw_var_peak = float(np.var(e))
    return {'rpdi': rpdi, 'sw_var_peak': sw_var_peak}

def extract_all_features(ents):
    e = np.array(ents, dtype=float)
    result = {'epr': float(e.mean()), 'trace_length': float(len(e))}
    gf = compute_spectral_features(ents)
    if gf is None: return None
    result.update(gf)
    result.update(compute_stft_features(ents))
    result.update(compute_time_domain(ents))
    return result

FEAT_NAMES = ['epr', 'trace_length', 'spectral_entropy', 'low_band_power', 'high_band_power',
              'hl_ratio', 'dominant_freq', 'spectral_centroid',
              'stft_max_high_power', 'stft_spectral_entropy', 'rpdi', 'sw_var_peak']
print(f'Feature set ({len(FEAT_NAMES)} signals): {FEAT_NAMES}')

Feature set (12 signals): ['epr', 'trace_length', 'spectral_entropy', 'low_band_power', 'high_band_power', 'hl_ratio', 'dominant_freq', 'spectral_centroid', 'stft_max_high_power', 'stft_spectral_entropy', 'rpdi', 'sw_var_peak']


In [3]:
# ── PART 1: Static comparison table (loads Phase 4 / Phase 5 pkl files) ────────
#
# No new inference — just reads already-saved results and formats a comparison table.
# Competitor AUC numbers are from their published papers:
#   RENT (arXiv:2505.22660)    — MATH-500/GPQA, Qwen2.5-Math models, same as Phase 4/5
#   LapEigvals (arXiv:2502.17598) — GSM8K + QA benchmarks, different models
#   LOS-Net (arXiv:2503.14043) — HotpotQA/Mistral-7B AUC=72.92% ← Phase 6 target

PHASE4_DIR = '/content/drive/MyDrive/epr_spectral_phase4'
PHASE5_DIR = '/content/drive/MyDrive/epr_spectral_phase5'

# Map: (model_short, dataset) → Phase4/5 pkl path
prior_runs = [
    {'label': 'MATH-500 / Qwen-7B   T=1.0 (Phase5)',  'tag': 'A2', 'dir': PHASE5_DIR,
     'run_key': 'Qwen2.5-Math-7B-Instruct__math500',  'pkl': 'phase5_results.pkl',
     'competitor': 'RENT', 'comp_auc': 'TBD', 'notes': 'Same model+dataset'},
    {'label': 'MATH-500 / Qwen-7B   T=1.5 (Phase4)',  'tag': 'A2-p4', 'dir': PHASE4_DIR,
     'run_key': 'Qwen2.5-Math-7B-Instruct__math500',  'pkl': 'phase4_results.pkl',
     'competitor': 'RENT', 'comp_auc': 'TBD', 'notes': 'Same model+dataset'},
    {'label': 'MATH-500 / Qwen-1.5B T=1.5 (Phase4)',  'tag': 'A1-p4', 'dir': PHASE4_DIR,
     'run_key': 'Qwen2.5-Math-1.5B-Instruct__math500', 'pkl': 'phase4_results.pkl',
     'competitor': 'RENT', 'comp_auc': 'TBD', 'notes': 'Same model+dataset'},
    {'label': 'GPQA / Mistral-7B    T=1.0 (Phase5)',  'tag': 'B1', 'dir': PHASE5_DIR,
     'run_key': 'Mistral-7B-Instruct-v0.2__gpqa',     'pkl': 'phase5_results.pkl',
     'competitor': 'RENT', 'comp_auc': 'TBD', 'notes': 'Same model+dataset'},
]

print('PART 1 — PRIOR RESULTS vs PUBLISHED BASELINES')
print('='*90)
print(f'{"Setting":<42} {"Our AUC":>10} {"Best subset":>32} {"Competitor":>10} {"Their AUC":>10}')
print('-'*90)

for r in prior_runs:
    pkl_path = os.path.join(r['dir'], r['run_key'], r['pkl'])
    if not os.path.exists(pkl_path):
        print(f"  {r['label']:<42}  MISSING: {pkl_path}")
        continue
    data = load_cache(pkl_path)
    sr   = data.get('subset_results', [])
    if sr:
        best  = sr[0]
        auc   = f"{100*best['auc']:.1f}% [{100*best['lo']:.1f},{100*best['hi']:.1f}]"
        subset = '+'.join(best['subset'])[:30]
    else:
        auc, subset = 'n/a', 'n/a'
    print(f"  {r['label']:<42}  {auc:>22}  {subset:<32}  {r['competitor']:>10}  {r['comp_auc']:>9}")

print('-'*90)
print(f"  {'HotpotQA / Mistral-7B T=1.0 (Phase6)':<42}  {'← Phase 6 result':>22}  {'':32}  {'LOS-Net':>10}  {'72.92%':>9}")
print()
print('NOTE: RENT and LapEigvals competitor AUCs marked TBD — fill in after reading their papers.')
print('LOS-Net HotpotQA/Mistral AUC=72.92% is the Phase 6 comparison target.')

PART 1 — PRIOR RESULTS vs PUBLISHED BASELINES
Setting                                       Our AUC                      Best subset Competitor  Their AUC
------------------------------------------------------------------------------------------
  MATH-500 / Qwen-7B   T=1.0 (Phase5)              90.0% [85.5,94.2]  trace_length+spectral_centroid          RENT        TBD
  MATH-500 / Qwen-7B   T=1.5 (Phase4)              96.6% [93.8,98.7]  epr+high_band_power+rpdi                RENT        TBD
  MATH-500 / Qwen-1.5B T=1.5 (Phase4)              88.3% [84.4,91.8]  epr+dominant_freq+rpdi                  RENT        TBD
  GPQA / Mistral-7B    T=1.0 (Phase5)              65.4% [57.3,73.4]  dominant_freq+stft_max_high_po          RENT        TBD
------------------------------------------------------------------------------------------
  HotpotQA / Mistral-7B T=1.0 (Phase6)              ← Phase 6 result                                       LOS-Net     72.92%

NOTE: RENT and LapEigvals compet

In [4]:
# ── PART 2: HotpotQA dataset loader and grading ────────────────────────────────
from datasets import load_dataset

def load_hotpotqa(n_samples=200):
    ds = load_dataset('hotpotqa/hotpot_qa', 'fullwiki', split='validation')
    samples = [ds[i] for i in range(min(n_samples, len(ds)))]
    print(f'Loaded {len(samples)} HotpotQA samples (fullwiki, validation).')
    return samples

def hotpotqa_prompt(row):
    q = row['question']
    return (
        f"You will answer a multi-hop question that requires finding two pieces "
        f"of information and connecting them.\n"
        f"Think step by step: first identify the intermediate fact, then use it "
        f"to answer the main question. Show your reasoning clearly.\n\n"
        f"Question: {q}\n\n"
        f"Provide your reasoning, then give your final answer on the last line."
    )

def normalize_answer(s):
    import string
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(c for c in s if c not in string.punctuation)
    return ' '.join(s.split())

def is_correct_hotpotqa(gen, gold):
    return normalize_answer(gold) in normalize_answer(gen)

print('HotpotQA loaders defined.')

HotpotQA loaders defined.


In [5]:
# ── Configuration ──────────────────────────────────────────────────────────────
BASE_DIR   = '/content/drive/MyDrive/epr_spectral_phase6'
os.makedirs(BASE_DIR, exist_ok=True)

CFG = {
    'model_id':   'mistralai/Mistral-7B-Instruct-v0.2',
    'dataset':    'hotpotqa',
    'n_samples':  200,
    'temp':       1.0,
    'max_new':    512,
}
CFG['model_short'] = CFG['model_id'].split('/')[-1]
CFG['run_key']     = f"{CFG['model_short']}__{CFG['dataset']}"
CFG['run_dir']     = os.path.join(BASE_DIR, CFG['run_key'])
os.makedirs(CFG['run_dir'], exist_ok=True)

CACHE_PATH   = os.path.join(CFG['run_dir'], 'inference_cache.pkl')
RESULTS_PATH = os.path.join(CFG['run_dir'], 'phase6_results.pkl')

print(f"Model:    {CFG['model_id']}")
print(f"Dataset:  HotpotQA fullwiki, {CFG['n_samples']} samples, T={CFG['temp']}")
print(f"Run dir:  {CFG['run_dir']}")
done = os.path.exists(RESULTS_PATH)
print(f"Status:   {'DONE (results file exists)' if done else 'pending inference'}")

Model:    mistralai/Mistral-7B-Instruct-v0.2
Dataset:  HotpotQA fullwiki, 200 samples, T=1.0
Run dir:  /content/drive/MyDrive/epr_spectral_phase6/Mistral-7B-Instruct-v0.2__hotpotqa
Status:   pending inference


In [6]:
# ── Inference loop (skips if results already saved) ────────────────────────────
if os.path.exists(RESULTS_PATH):
    print('Results already saved — skipping inference. Run feature extraction cell next.')
else:
    hotpotqa_data = load_hotpotqa(CFG['n_samples'])
    cache         = load_cache(CACHE_PATH)
    remaining     = [i for i in range(CFG['n_samples']) if not cache.get(i, {}).get('done')]
    print(f"Cache: {CFG['n_samples'] - len(remaining)}/{CFG['n_samples']} done. Remaining: {len(remaining)}")

    if remaining:
        model, tokenizer = load_model(CFG['model_id'])
        for i in tqdm(remaining, desc='Phase6 inference'):
            try:
                row    = hotpotqa_data[i]
                prompt = hotpotqa_prompt(row)
                full_text, all_ents = generate_full(
                    model, tokenizer, prompt, CFG['temp'], max_new_tokens=CFG['max_new'])
                correct = is_correct_hotpotqa(full_text, row['answer'])
                cache[i] = {'done': True, 'full_text': full_text,
                            'all_entropies': all_ents, 'correct': correct,
                            'gold': row['answer']}
            except Exception as ex:
                print(f'  Error {i}: {ex}')
                cache[i] = {'done': False}
            if i % 20 == 0: save_cache(cache, CACHE_PATH)
            free_memory()
        save_cache(cache, CACHE_PATH)
        del model, tokenizer; free_memory()

    n_done    = sum(1 for i in range(CFG['n_samples']) if cache.get(i, {}).get('done'))
    n_correct = sum(1 for i in range(CFG['n_samples'])
                    if cache.get(i, {}).get('done') and cache[i].get('correct'))
    print(f'Inference complete: {n_done} samples, {n_correct} correct ({n_correct/max(n_done,1):.1%})')

README.md: 0.00B [00:00, ?B/s]

fullwiki/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

fullwiki/validation-00000-of-00001.parqu(…):   0%|          | 0.00/28.0M [00:00<?, ?B/s]

fullwiki/test-00000-of-00001.parquet:   0%|          | 0.00/27.6M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded mistralai/Mistral-7B-Instruct-v0.2


Phase6 inference: 100%|██████████| 200/200 [33:34<00:00, 10.07s/it]


Inference complete: 200 samples, 68 correct (34.0%)


In [7]:
# ── Feature extraction (full response — no trace/answer split) ─────────────────
#
# Design decision: the full 50–200 token response IS the signal.
# No split needed — marker appeared 0–2% of time in prior experiments.

cache    = load_cache(CACHE_PATH)
idx_ok   = sorted(i for i in range(CFG['n_samples'])
                  if cache.get(i, {}).get('done') and cache[i].get('all_entropies'))

labels, feat_list, n_toks, raw_ents = [], [], [], []
for i in idx_ok:
    v    = cache[i]
    ents = v['all_entropies']
    if len(ents) < 8: continue
    feats = extract_all_features(ents)
    if feats is None: continue
    labels.append(int(v['correct']))
    feat_list.append(feats)
    n_toks.append(len(ents))
    raw_ents.append(ents)

labels      = np.array(labels)
n_toks      = np.array(n_toks)
feat_arrays = {name: np.array([f[name] for f in feat_list]) for name in FEAT_NAMES}

print(f'Samples:      {len(labels)}')
print(f'Correct:      {int(labels.sum())} ({labels.mean():.1%})')
print(f'Avg trace:    {n_toks.mean():.1f} tokens  (min={n_toks.min()}, max={n_toks.max()})')
print(f'Traces < 32:  {(n_toks < 32).sum()}  (stft_max_high_power → 0 for these)')

Samples:      200
Correct:      68 (34.0%)
Avg trace:    249.8 tokens  (min=61, max=512)
Traces < 32:  0  (stft_max_high_power → 0 for these)


In [8]:
# ── Window ablation: sw_var_peak with w ∈ {3, 5, 7, 9, 16} ────────────────────
#
# LSC paper (arXiv:2601.19918) confirms w=2–3 optimal on NQ/TriviaQA/SQuAD/CoQA.
# sw_step=1 for token-by-token sensitivity (no striding artefacts).
# Runs on cached entropy arrays — no re-inference needed.

WINDOW_SIZES = [3, 5, 7, 9, 16]

def sw_var_peak_with_window(ents, sw_window, sw_step=1):
    e = np.array(ents, dtype=float)
    if len(e) >= sw_window:
        sw_vars = [np.var(e[i:i+sw_window])
                   for i in range(0, len(e)-sw_window+1, sw_step)]
        return float(np.max(sw_vars))
    else:
        return float(np.var(e))

print(f'Window ablation  (sw_step=1, n={len(labels)} samples)')
print(f'{"Window":>8}  {"AUC":>8}  {"95% CI":>16}  {"Eligible traces":>16}')
print('-' * 56)

ablation_results = []
for w in WINDOW_SIZES:
    sw_vals  = np.array([sw_var_peak_with_window(e, w) for e in raw_ents])
    n_elig   = sum(1 for e in raw_ents if len(e) >= w)
    ap, lop, hip = boot_auc(labels,  sw_vals)
    an, lon, hin = boot_auc(labels, -sw_vals)
    if ap >= an: auc, lo, hi, sign = ap, lop, hip, '+'
    else:        auc, lo, hi, sign = an, lon, hin, '−'
    ablation_results.append({'w': w, 'auc': auc, 'lo': lo, 'hi': hi, 'sign': sign,
                             'vals': sw_vals})
    print(f'  w={w:<4}  {100*auc:>6.1f}%  [{100*lo:>5.1f}, {100*hi:>5.1f}]  {n_elig}/{len(raw_ents)}')

best_w = max(ablation_results, key=lambda x: x['auc'])
print(f'\nBest window: w={best_w["w"]}  AUC={100*best_w["auc"]:.1f}%')

# Override sw_var_peak in feat_arrays with best-window version
feat_arrays['sw_var_peak'] = best_w['vals'] * (1 if best_w['sign'] == '+' else -1)
print(f'feat_arrays["sw_var_peak"] updated → w={best_w["w"]} (sign={best_w["sign"]})')

Window ablation  (sw_step=1, n=200 samples)
  Window       AUC            95% CI   Eligible traces
--------------------------------------------------------
  w=3       52.7%  [ 44.4,  61.7]  200/200
  w=5       51.1%  [ 42.6,  59.6]  200/200
  w=7       51.5%  [ 42.8,  60.1]  200/200
  w=9       52.7%  [ 44.1,  61.3]  200/200
  w=16      53.6%  [ 44.7,  62.6]  200/200

Best window: w=16  AUC=53.6%
feat_arrays["sw_var_peak"] updated → w=16 (sign=+)


In [9]:
# ── Individual AUCs + Spearman ρ matrix ────────────────────────────────────────
auc_map, sign_map, rows = {}, {}, []
for name in FEAT_NAMES:
    a_p, lo_p, hi_p = boot_auc(labels,  feat_arrays[name])
    a_n, lo_n, hi_n = boot_auc(labels, -feat_arrays[name])
    if a_p >= a_n: auc, lo, hi, sign = a_p, lo_p, hi_p, +1
    else:          auc, lo, hi, sign = a_n, lo_n, hi_n, -1
    auc_map[name] = auc; sign_map[name] = sign
    rows.append((auc, name, lo, hi))
rows.sort(reverse=True)

print('Individual AUCs — HotpotQA / Mistral-7B T=1.0')
print(f'{"Signal":<26}  {"AUC":>8}  {"95% CI":>16}  {"sign":>5}')
print('-' * 62)
for auc, name, lo, hi in rows:
    print(f'  {name:<26} {100*auc:>7.1f}%  [{100*lo:>5.1f}, {100*hi:>5.1f}]  {sign_map[name]:>+4d}')

# Spearman ρ matrix (oriented)
oriented = {name: feat_arrays[name] * sign_map[name] for name in FEAT_NAMES}
rho_mat  = {}
for a, b in itertools.combinations_with_replacement(FEAT_NAMES, 2):
    rho, _ = spearmanr(oriented[a], oriented[b])
    rho_mat[(a, b)] = rho_mat[(b, a)] = rho

print('\nSpearman ρ for pairs with |ρ| > 0.60 (oriented):')
printed = set()
for (a, b), rho in sorted(rho_mat.items(), key=lambda x: -abs(x[1])):
    if a >= b or (a, b) in printed: continue
    if abs(rho) > 0.60:
        flag = ' ← BLOCKED (≥0.75)' if abs(rho) >= 0.75 else ''
        print(f'  {a:<26} × {b:<26}  ρ={rho:+.3f}{flag}')
        printed.add((a, b))

Individual AUCs — HotpotQA / Mistral-7B T=1.0
Signal                           AUC            95% CI   sign
--------------------------------------------------------------
  spectral_entropy              55.0%  [ 45.9,  63.7]    -1
  trace_length                  54.5%  [ 45.6,  63.5]    -1
  epr                           53.7%  [ 44.6,  62.9]    +1
  sw_var_peak                   53.6%  [ 44.7,  62.6]    +1
  dominant_freq                 53.3%  [ 44.2,  61.9]    +1
  high_band_power               51.3%  [ 43.1,  59.3]    -1
  stft_spectral_entropy         51.1%  [ 42.0,  59.8]    -1
  rpdi                          51.0%  [ 42.7,  60.2]    -1
  stft_max_high_power           51.0%  [ 42.3,  58.5]    +1
  spectral_centroid             50.9%  [ 42.3,  59.1]    -1
  hl_ratio                      50.2%  [ 42.2,  58.6]    +1
  low_band_power                50.0%  [ 41.1,  58.9]    +1

Spearman ρ for pairs with |ρ| > 0.60 (oriented):
  spectral_entropy           × trace_length                

In [10]:
# ── Nadler combinatorial fusion ────────────────────────────────────────────────
print('Running Nadler fusion...')
best_auc, best_lo, best_hi, best_subset = best_nadler_on(
    feat_arrays, FEAT_NAMES, labels, max_size=4, label='Phase6')

print()
if best_subset:
    print(f'Best fusion:  {"+".join(best_subset)}')
    print(f'AUC:          {100*best_auc:.1f}%  [{100*best_lo:.1f}, {100*best_hi:.1f}]')
    print(f'LOS-Net ref:  72.92%  (HotpotQA/Mistral-7B, supervised)')
    diff = (best_auc - 0.7292) * 100
    print(f'Δ vs LOS-Net: {diff:+.1f} pp')
else:
    print('No valid fusion subset found.')

Running Nadler fusion...
  [Phase6] 12 features, 12 informative, max_size=4 → 781 raw combos
    size=2: 66 combos, 60 passed ρ-filter, best so far=58.9%
    size=3: 220 combos, 164 passed ρ-filter, best so far=59.4%
    size=4: 495 combos, 269 passed ρ-filter, best so far=59.5%
  [Phase6] done — checked=493, skipped(ρ)=288, best=59.5%

Best fusion:  spectral_entropy+low_band_power+stft_spectral_entropy+sw_var_peak
AUC:          59.5%  [51.1, 66.9]
LOS-Net ref:  72.92%  (HotpotQA/Mistral-7B, supervised)
Δ vs LOS-Net: -13.4 pp


In [11]:
# ── Decision gates G0–G6 ───────────────────────────────────────────────────────
LOS_NET_AUC = 0.7292

best_individual_auc = max(auc_map.values())
ci_lower            = best_lo
best_window         = best_w['w']

gates = [
    ('G0', 'Sufficient samples',       len(labels) >= 150,
     f'len(labels)={len(labels)} ≥ 150',
     'Run more samples to proceed'),
    ('G1', 'Spectral structure exists', best_individual_auc > 0.55,
     f'best individual AUC={100*best_individual_auc:.1f}% > 55%',
     'No spectral transfer to HotpotQA'),
    ('G2', 'Method works on multi-hop', best_auc > 0.65,
     f'best fusion AUC={100*best_auc:.1f}% > 65%',
     'Spectral features are math-specific only'),
    ('G3', 'Beat LOS-Net on HotpotQA', best_auc > LOS_NET_AUC,
     f'best fusion={100*best_auc:.1f}% > LOS-Net {100*LOS_NET_AUC:.2f}% (supervised)',
     'LOS-Net leads on factual QA — strong baseline'),
    ('G4', 'sw_var_peak transfers',     auc_map.get('sw_var_peak', 0) > 0.60,
     f'sw_var_peak AUC={100*auc_map.get("sw_var_peak",0):.1f}% > 60%',
     'sw_var_peak is math-specific, does not transfer'),
    ('G5', 'Window ablation confirms LSC', best_window <= 9,
     f'optimal w*={best_window} ≤ 9 (LSC: w=2–3 on short factual QA)',
     'Window size does not matter for HotpotQA traces'),
    ('G6', 'Statistically reliable',    ci_lower > 0.55,
     f'95% CI lower={100*ci_lower:.1f}% > 55%',
     'Too few samples — bootstrap CI too wide'),
]

print('DECISION GATES — Phase 6')
print('='*80)
n_pass = 0
for gate_id, name, passed, pass_msg, fail_msg in gates:
    status = 'PASS ✓' if passed else 'FAIL ✗'
    msg    = pass_msg if passed else fail_msg
    print(f'{gate_id}  {name:<32}  [{status}]  {msg}')
    if passed: n_pass += 1
print(f'\n{n_pass}/{len(gates)} gates passed.')

if n_pass == len(gates):
    print('→ All gates passed. Strong positive result.')
elif n_pass >= 4:
    print('→ Majority passed. Method transfers with partial success.')
elif n_pass >= 2:
    print('→ Partial transfer. Spectral structure exists but scope is limited.')
else:
    print('→ Weak result. Spectral features do not transfer to HotpotQA.')

DECISION GATES — Phase 6
G0  Sufficient samples                [PASS ✓]  len(labels)=200 ≥ 150
G1  Spectral structure exists         [PASS ✓]  best individual AUC=55.0% > 55%
G2  Method works on multi-hop         [FAIL ✗]  Spectral features are math-specific only
G3  Beat LOS-Net on HotpotQA          [FAIL ✗]  LOS-Net leads on factual QA — strong baseline
G4  sw_var_peak transfers             [FAIL ✗]  sw_var_peak is math-specific, does not transfer
G5  Window ablation confirms LSC      [FAIL ✗]  Window size does not matter for HotpotQA traces
G6  Statistically reliable            [FAIL ✗]  Too few samples — bootstrap CI too wide

2/7 gates passed.
→ Partial transfer. Spectral structure exists but scope is limited.


In [12]:
# ── Final comparison table + save summary JSON ─────────────────────────────────
import json

print('FINAL COMPARISON TABLE — Phase 6')
print('='*90)
print(f'{"Dataset / Model":<42} {"Our AUC":>12} {"Method":>22} {"Competitor":>10} {"Their AUC":>10}')
print('-'*90)

# Static rows from Part 1
static_rows = [
    ('MATH-500 / Qwen-7B T=1.0',   '90.0%', 'Spectral Nadler', 'RENT',      'TBD'),
    ('MATH-500 / Qwen-7B T=1.5',   '96.6%', 'Spectral Nadler', 'RENT',      'TBD'),
    ('MATH-500 / Qwen-1.5B T=1.5', '88.3%', 'Spectral Nadler', 'RENT',      'TBD'),
    ('GPQA / Mistral-7B T=1.0',    '65.4%', 'Spectral Nadler', 'RENT',      'TBD'),
    ('GSM8K / Qwen-1.5B',          '75.9%', 'Spectral Nadler', 'LapEigvals','TBD'),
]
for label, our_auc, method, comp, their_auc in static_rows:
    print(f'  {label:<42} {our_auc:>12} {method:>22} {comp:>10} {their_auc:>10}')

# Phase 6 live row
p6_auc_str = f'{100*best_auc:.1f}% [{100*best_lo:.1f},{100*best_hi:.1f}]'
p6_subset  = '+'.join(best_subset) if best_subset else 'n/a'
print(f'  {"HotpotQA / Mistral-7B T=1.0":<42} {p6_auc_str:>12} {"Spectral Nadler":>22} {"LOS-Net":>10} {"72.92%":>10}')
print('='*90)
print(f'Best subset (HotpotQA): {p6_subset}')
print(f'LOS-Net Δ: {(best_auc - LOS_NET_AUC)*100:+.1f} pp')

# Save summary
summary = {
    'phase': 6,
    'model': CFG['model_id'],
    'dataset': 'hotpotqa_fullwiki_validation',
    'n_samples': len(labels),
    'accuracy': float(labels.mean()),
    'avg_trace_len': float(n_toks.mean()),
    'best_fusion_auc': float(best_auc),
    'best_fusion_lo':  float(best_lo),
    'best_fusion_hi':  float(best_hi),
    'best_subset': list(best_subset) if best_subset else [],
    'losnet_auc': LOS_NET_AUC,
    'delta_vs_losnet': float(best_auc - LOS_NET_AUC),
    'individual_aucs': {k: float(v) for k, v in auc_map.items()},
    'window_ablation': [{'w': r['w'], 'auc': float(r['auc']),
                          'lo': float(r['lo']), 'hi': float(r['hi'])} for r in ablation_results],
    'best_window': int(best_w['w']),
    'gates': {g[0]: bool(g[2]) for g in gates},
    'n_gates_passed': n_pass,
}

summary_path = os.path.join(BASE_DIR, 'phase6_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

save_cache({'tag': 'phase6', 'model': CFG['model_short'], 'dataset': 'hotpotqa',
            'temp': CFG['temp'], 'n_samples': len(labels),
            'accuracy': float(labels.mean()), 'avg_trace': float(n_toks.mean()),
            'auc_map': auc_map, 'sign_map': sign_map, 'rho_mat': rho_mat,
            'subset_results': [{'subset': list(best_subset), 'auc': float(best_auc),
                                 'lo': float(best_lo), 'hi': float(best_hi)}] if best_subset else [],
            'feat_names': FEAT_NAMES,
            'raw_labels': labels.tolist(),
            'feat_arrays': {k: v.tolist() for k, v in feat_arrays.items()},
            'ablation': summary['window_ablation']},
           RESULTS_PATH)

print(f'\nSaved: {summary_path}')
print(f'Saved: {RESULTS_PATH}')

FINAL COMPARISON TABLE — Phase 6
Dataset / Model                                 Our AUC                 Method Competitor  Their AUC
------------------------------------------------------------------------------------------
  MATH-500 / Qwen-7B T=1.0                          90.0%        Spectral Nadler       RENT        TBD
  MATH-500 / Qwen-7B T=1.5                          96.6%        Spectral Nadler       RENT        TBD
  MATH-500 / Qwen-1.5B T=1.5                        88.3%        Spectral Nadler       RENT        TBD
  GPQA / Mistral-7B T=1.0                           65.4%        Spectral Nadler       RENT        TBD
  GSM8K / Qwen-1.5B                                 75.9%        Spectral Nadler LapEigvals        TBD
  HotpotQA / Mistral-7B T=1.0                59.5% [51.1,66.9]        Spectral Nadler    LOS-Net     72.92%
Best subset (HotpotQA): spectral_entropy+low_band_power+stft_spectral_entropy+sw_var_peak
LOS-Net Δ: -13.4 pp

Saved: /content/drive/MyDrive/epr_spectra